# Prime-power semigroup diagnostics (6×6→Schur→Cayley)

This notebook implements the concrete diagnostic plan discussed:

1. **Matrix-log generator existence**: check whether $G_{p,k} := rac{1}{k}og S_{p,k}$ is stable in $k$.
2. **Eigenphase linearity**: check whether eigenphases scale $	heta_{p,k,j} pprox k	heta_{p,1,j}.$
3. **Cayley-inverse additivity**: pull back $S$ to $ambda$ and test $ambda_{p,k} pprox kambda_{p,1}$ and a best-scalar fit.
4. **Upstairs reconstruction**: build an upstream generator $T_p$ from the $k=1$ seed and test whether $athcal R(T_p^k)$ matches the observed packets.
5. **Slope coherence**: fit scaling laws vs $p$ and check linearity in $k$.

All diagnostics use the same local reduction map $athcal R$ used in the FE tools:
$$X 	o A=athrm{Cayley}(X) 	o B(A,A^harp) 	o ambda=athrm{Schur}(B) 	o S=athrm{Cayley}(ambda).$$

## Suggested thresholds (heuristic)
These are intentionally conservative and meant for *comparative* interpretation:

- **Good semigroup evidence**: median error < 0.05; 90th percentile < 0.15
- **Mixed/coordinate distortion**: median 0.05–0.25
- **No semigroup downstairs**: median > 0.25

Apply this to each metric: generator stability, phase-linearity, Lambda additivity, and reconstruction error.

---
**Outputs**: this notebook writes one wide CSV with per-(p,k) metrics, plus a short summary CSV aggregated by k and p_mode.

In [ ]:
from __future__ import annotations

import math
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

# Import simulator module from tools/ (tools is not a package)
TOOLS_DIR = Path.cwd() / 'tools'
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))
import six_by_six_prime_tower_sim as sim  # type: ignore

np.set_printoptions(precision=4, suppress=True)

## Configuration
Keep this aligned with your fixed-gauge FE discrimination settings so the diagnostics are on the same slice.

In [ ]:
# Prime set and tower depth
PRIMES = [2,3,5,7,11,13,17,19,23,29,31,37]
K_MAX = 3

# Deformation (match FE tooling convention: v=-u)
U = 0.2
V = -U

# Fixed gauge (match your recent discrimination runs)
BOUNDARY = (0, 5)
SCHUR_SIGN = '+'

# Local construction settings
SHARP_MODE = 'conj_transpose'
X_MODE = 'det1'
X_GAMMA = 1.0
SCATTERING_MODE = 'i_pm_lambda'

# Which p-injection to test inside the *local* diagonal factor D
P_MODES = ['p', 'logp', 'invp', 'p1_over_p', 'p_over_p1', 'p_minus1_over_p']

OUT_DIR = Path('runs_fe')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_WIDE = OUT_DIR / 'semigroup_diagnostics_wide.csv'
OUT_SUMMARY = OUT_DIR / 'semigroup_diagnostics_summary.csv'

(OUT_WIDE, OUT_SUMMARY)

## Helper functions
These implement matrix-log via eigen decomposition (2×2), eigenvalue pairing across k, and inverse-Cayley $Sapstoambda$ for the `i_pm_lambda` convention.

In [ ]:
def fro_norm(M: np.ndarray) -> float:
    return float(np.linalg.norm(np.asarray(M, dtype=np.complex128), ord='fro'))


def rel_fro(A: np.ndarray, B: np.ndarray, *, den_floor: float = 1e-12) -> float:
    A = np.asarray(A, dtype=np.complex128)
    B = np.asarray(B, dtype=np.complex128)
    num = float(np.linalg.norm(A - B, ord='fro'))
    den = max(float(den_floor), float(np.linalg.norm(B, ord='fro')))
    return float(num / den)


def inv_cayley_i_pm_lambda(S: np.ndarray) -> np.ndarray:
    # If S = (I - iΛ)(I + iΛ)^{-1}, then Λ = i (S + I)^{-1} (S - I).
    S = np.asarray(S, dtype=np.complex128)
    I = np.eye(2, dtype=np.complex128)
    return (1j * np.linalg.solve(S + I, S - I)).astype(np.complex128)


def eigvals_2x2(S: np.ndarray) -> tuple[complex, complex]:
    w = np.linalg.eigvals(np.asarray(S, dtype=np.complex128))
    return (complex(w[0]), complex(w[1]))


def pair_eigs(prev: tuple[complex, complex], cur: tuple[complex, complex]) -> tuple[complex, complex]:
    # Pair cur eigenvalues to prev by minimal total distance.
    a0, b0 = prev
    a1, b1 = cur
    d_same = abs(a1 - a0) + abs(b1 - b0)
    d_swap = abs(b1 - a0) + abs(a1 - b0)
    return (a1, b1) if d_same <= d_swap else (b1, a1)


def principal_log_unit_circle(z: complex) -> complex:
    # For near-unitary eigenvalues: log(z) ≈ i*Arg(z) with principal branch.
    ang = math.atan2(float(np.imag(z)), float(np.real(z)))
    return 1j * float(ang)


def logm_2x2_via_eigs(S: np.ndarray) -> np.ndarray:
    # 2×2 diagonalizable log via eigendecomposition; tailored for near-unitary S.
    S = np.asarray(S, dtype=np.complex128)
    w, V = np.linalg.eig(S)
    # Use unit-circle principal log for each eigenvalue.
    L = np.diag([principal_log_unit_circle(complex(wi)) for wi in w]).astype(np.complex128)
    Vinv = np.linalg.inv(V)
    return (V @ L @ Vinv).astype(np.complex128)


def best_scalar_fro(A: np.ndarray, B: np.ndarray) -> complex:
    # argmin_a ||A - a B||_F: a = <A,B> / <B,B> using Frobenius inner product
    A = np.asarray(A, dtype=np.complex128)
    B = np.asarray(B, dtype=np.complex128)
    num = complex(np.vdot(B, A))
    den = complex(np.vdot(B, B))
    if abs(den) <= 1e-300:
        return complex(0.0)
    return num / den


def phase(z: complex) -> float:
    return float(math.atan2(float(np.imag(z)), float(np.real(z))))


def unwrap_like(prev_theta: float, theta: float) -> float:
    # shift theta by 2πm to be closest to prev_theta
    dt = float(theta - prev_theta)
    m = round(dt / (2.0 * math.pi))
    return float(theta - (2.0 * math.pi) * m)

## Build observed packets $S_{p,k}$ and compute diagnostics
This uses the same internal functions as the FE tooling:
- `sim._local_blocks_for_prime_power` (build X,A,A^sharp)
- `sim._bulk_B_from_A` (6×6 bulk)
- `sim._schur_complement_Lambda` (2×2 boundary Schur complement)
- `sim._scattering_from_Lambda` (2×2 scattering)

We compute, per (p,k,p_mode):
- `rel_pow_closure`: $S_{p,k}-(S_{p,1})^k_F / S_{p,k}_F$
- `gen_log_stability`: $G_{p,k}-G_{p,1}_F / G_{p,1}_F$ with $G_{p,k}=(1/k)og S_{p,k}$
- `phase_linearity`: eigenphase errors vs k·phase(k=1)
- `lambda_add_err`: $ambda_{p,k}-kambda_{p,1}_F / (1+ambda_{p,k}_F)$
- `lambda_axis_err`: best scalar-fit axis error $ambda_{p,k}-aambda_{p,1}_F / (1+ambda_{p,k}_F)$
- `recon_err_Xgen`: reconstruction error where the upstream generator is $T_p := X_{p,1}$ and $X_{pred,k}:=T_p^k$ then reduced.


In [ ]:
rows: list[dict] = []

for p_mode in P_MODES:
    for p in PRIMES:
        # Observed tower
        obs: dict[int, dict] = {}
        for k in range(1, K_MAX + 1):
            X, A, Ash = sim._local_blocks_for_prime_power(
                int(p), int(k),
                sharp_mode=SHARP_MODE,
                x_mode=X_MODE,
                x_gamma=float(X_GAMMA),
                x_shear=float(U),
                x_lower=float(V),
                p_mode=str(p_mode),
            )
            B = sim._bulk_B_from_A(A, Ash)
            Lam = sim._schur_complement_Lambda(B, boundary=BOUNDARY, sign=SCHUR_SIGN)
            S = sim._scattering_from_Lambda(Lam, mode=SCATTERING_MODE)
            obs[int(k)] = {
: X, 
: A, 
: Lam, 
: S}

        S1 = np.asarray(obs[1][
], dtype=np.complex128)
        X1 = np.asarray(obs[1][
], dtype=np.complex128)
        Lam1 = inv_cayley_i_pm_lambda(S1) if SCATTERING_MODE == 'i_pm_lambda' else obs[1][
]
        logS1 = logm_2x2_via_eigs(S1)
        G1 = logS1  # since k=1
        w1 = eigvals_2x2(S1)
        th1 = (phase(w1[0]), phase(w1[1]))
        # Make phase ordering deterministic by sorting by angle.
        if th1[1] < th1[0]:
            th1 = (th1[1], th1[0])
            w1 = (w1[1], w1[0])

        for k in range(1, K_MAX + 1):
            S = np.asarray(obs[k][
], dtype=np.complex128)
            Lam = inv_cayley_i_pm_lambda(S) if SCATTERING_MODE == 'i_pm_lambda' else np.asarray(obs[k][
], dtype=np.complex128)
            X = np.asarray(obs[k][
], dtype=np.complex128)

            # 0) Raw closure S_{p,k} vs (S_{p,1})^k
            Spow = np.eye(2, dtype=np.complex128)
            for _ in range(k):
                Spow = (Spow @ S1).astype(np.complex128)
            rel_pow_closure = rel_fro(S, Spow)

            # 1) Generator existence via matrix logs: G_{p,k}=(1/k)log S_{p,k}
            logSk = logm_2x2_via_eigs(S)
            Gk = (logSk / float(k)).astype(np.complex128)
            gen_log_stability = rel_fro(Gk, G1)

            # 2) Eigenphase linearity
            wk = eigvals_2x2(S)
            # Pair to k=1 ordering.
            wk = pair_eigs(w1, wk)
            thk_raw = (phase(wk[0]), phase(wk[1]))
            # Unwrap each phase to be near k*th1.
            thk = (unwrap_like(k * th1[0], thk_raw[0]), unwrap_like(k * th1[1], thk_raw[1]))
            phase_err0 = abs(thk[0] - k * th1[0]) / (1.0 + abs(k * th1[0]))
            phase_err1 = abs(thk[1] - k * th1[1]) / (1.0 + abs(k * th1[1]))
            phase_linearity = float(0.5 * (phase_err0 + phase_err1))

            # 3) Cayley-inverse additivity (Lambda layer)
            lambda_add_err = rel_fro(Lam, float(k) * Lam1, den_floor=1.0)
            a = best_scalar_fro(Lam, Lam1)
            lambda_axis_err = rel_fro(Lam, a * Lam1, den_floor=1.0)
            a_re = float(np.real(a))
            a_im = float(np.imag(a))

            # 4) Upstairs reconstruction using X-generator: T_p := X_{p,1}, X_pred = T_p^k
            Xpred = np.eye(2, dtype=np.complex128)
            for _ in range(k):
                Xpred = (Xpred @ X1).astype(np.complex128)
            Apred = sim._cayley(Xpred)
            Ashpred = sim._symplectic_partner(Apred, mode=SHARP_MODE)
            Bpred = sim._bulk_B_from_A(Apred, Ashpred)
            Lampred = sim._schur_complement_Lambda(Bpred, boundary=BOUNDARY, sign=SCHUR_SIGN)
            Spred = sim._scattering_from_Lambda(Lampred, mode=SCATTERING_MODE)
            recon_err_Xgen = rel_fro(Spred, S, den_floor=1.0)

            # 5) Scale observables for later slope coherence fits
            I2 = np.eye(2, dtype=np.complex128)
            dev_S_pm_I = float(min(fro_norm(S - I2), fro_norm(S + I2)))

            rows.append({
                'p_mode': str(p_mode),
                'p': int(p),
                'k': int(k),
                'u': float(U),
                'boundary': f'{BOUNDARY[0]},{BOUNDARY[1]}',
                'schur_sign': str(SCHUR_SIGN),
                'x_mode': str(X_MODE),
                'x_gamma': float(X_GAMMA),
                'sharp_mode': str(SHARP_MODE),
                'scattering_mode': str(SCATTERING_MODE),
                'unit_def_S': float(sim._unitarity_defect(S)),
                'herm_def_Lam': float(sim._hermitian_defect(Lam)),
                'dev_S_pm_I': float(dev_S_pm_I),
                'rel_pow_closure': float(rel_pow_closure),
                'gen_log_stability': float(gen_log_stability),
                'phase_linearity': float(phase_linearity),
                'lambda_add_err': float(lambda_add_err),
                'lambda_axis_err': float(lambda_axis_err),
                'lambda_axis_a_re': float(a_re),
                'lambda_axis_a_im': float(a_im),
                'recon_err_Xgen': float(recon_err_Xgen),
            })
df = pd.DataFrame(rows)
df.to_csv(OUT_WIDE, index=False)
OUT_WIDE

## Summary tables
Aggregates by (p_mode,k) with robust stats to drive your decision tree.

In [ ]:
def robust_summary(df: pd.DataFrame, col: str) -> dict:
    x = df[col].to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return {'median': float('nan'), 'p90': float('nan'), 'mean': float('nan')}
    return {
        'median': float(np.median(x)),
        'p90': float(np.quantile(x, 0.90)),
        'mean': float(np.mean(x)),
    }

metrics = ['rel_pow_closure', 'gen_log_stability', 'phase_linearity', 'lambda_add_err', 'lambda_axis_err', 'recon_err_Xgen']

summ_rows = []
for (p_mode, k), g in df.groupby(['p_mode', 'k']):
    row = {'p_mode': p_mode, 'k': int(k), 'n_primes': int(g.shape[0])}
    for m in metrics:
        s = robust_summary(g, m)
        row[f'{m}_median'] = s['median']
        row[f'{m}_p90'] = s['p90']
        row[f'{m}_mean'] = s['mean']
    summ_rows.append(row)
summary = pd.DataFrame(summ_rows).sort_values(['p_mode', 'k'])
summary.to_csv(OUT_SUMMARY, index=False)
summary

## Slope coherence vs prime (per k)
Fits $og(athrm{dev}) = a_k + b_kog p$ and then checks whether $b_k$ is approximately linear in k.

In [ ]:
def fit_loglog_slope(p: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    p = np.asarray(p, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    m = np.isfinite(p) & np.isfinite(y) & (p > 0) & (y > 0)
    p = p[m]
    y = y[m]
    if p.size < 2:
        return float('nan'), float('nan')
    X = np.log(p)
    Y = np.log(y)
    slope, intercept = np.polyfit(X, Y, 1)
    Yhat = slope * X + intercept
    ss_res = float(np.sum((Y - Yhat) ** 2))
    ss_tot = float(np.sum((Y - float(np.mean(Y))) ** 2))
    r2 = 1.0 - (ss_res / (ss_tot + 1e-300))
    return float(slope), float(r2)

slope_rows = []
for (p_mode, k), g in df.groupby(['p_mode', 'k']):
    slope, r2 = fit_loglog_slope(g['p'].to_numpy(), g['dev_S_pm_I'].to_numpy())
    slope_rows.append({'p_mode': p_mode, 'k': int(k), 'slope_bk': slope, 'r2': r2})

slopes = pd.DataFrame(slope_rows).sort_values(['p_mode', 'k'])
slopes

In [ ]:
# Check b_k ~ alpha + beta*k per p_mode (simple linear fit)
lin_rows = []
for p_mode, g in slopes.groupby('p_mode'):
    kk = g['k'].to_numpy(dtype=float)
    bb = g['slope_bk'].to_numpy(dtype=float)
    m = np.isfinite(kk) & np.isfinite(bb)
    kk = kk[m]; bb = bb[m]
    if kk.size < 2:
        continue
    beta, alpha = np.polyfit(kk, bb, 1)  # bb ~ beta*k + alpha
    bb_hat = beta * kk + alpha
    ss_res = float(np.sum((bb - bb_hat) ** 2))
    ss_tot = float(np.sum((bb - float(np.mean(bb))) ** 2))
    r2 = 1.0 - ss_res / (ss_tot + 1e-300)
    lin_rows.append({'p_mode': p_mode, 'alpha': float(alpha), 'beta': float(beta), 'r2': float(r2)})
pd.DataFrame(lin_rows).sort_values('p_mode')